In [1]:
import numpy as np
import pandas as pd
import scanpy as sc
import anndata as ad
import os 
import json
from tqdm import tqdm
from scipy.sparse import isspmatrix, coo_matrix
from multiprocessing import Pool, cpu_count

In [ ]:


# 全局变量（只读）
X_coo = None

def init_worker(matrix):
    global X_coo
    X_coo = matrix

def process_cell_chunk(args):
    chunk, peaks, cells = args
    results = []
    for i in chunk:
        mask = (X_coo.row == i)
        for j, count in zip(X_coo.col[mask], X_coo.data[mask]):
            chrom, start, end = peaks.iloc[j]
            results.append((chrom, int(start), int(end), cells[i], int(count)))
    return results

def adata_to_count_fragments_parallel(adata, output_dir, chr = 'chr_m', start = 'start_m', end = 'end_m', n_cpu=None):
    global X_coo

    peaks = adata.var[[chr, start, end]].copy()
    peaks[start] = peaks[start].astype(int)
    peaks[end] = peaks[end].astype(int)
    cells = adata.obs_names.tolist()

    X = adata.X #if not hasattr(adata.X, "toarray") else adata.X.toarray()
    X_coo = coo_matrix(X)

    # Step 3: 分块
    n_cpu = n_cpu or min(cpu_count(), 16)
    cell_indices = np.arange(X_coo.shape[0])
    chunks = np.array_split(cell_indices, n_cpu)

    pool_args = [(chunk, peaks, cells) for chunk in chunks]

    # Step 4: 多进程 + 进度条
    with Pool(n_cpu, initializer=init_worker, initargs=(X_coo,)) as pool:
        results = []
        for result in tqdm(pool.imap_unordered(process_cell_chunk, pool_args),
                           desc="Processing fragments",
                           total=len(pool_args)):
            results.append(result)

    # 扁平化
    fragments = [item for sublist in results for item in sublist]

    # 写入 TSV
    df = pd.DataFrame(fragments, columns=['chrom', 'start', 'end', 'cell_barcode', 'count'])
    df = df.sort_values(['chrom', 'start'])

    os.makedirs(output_dir, exist_ok=True)
    df.to_csv(os.path.join(output_dir, 'fragment.tsv'), sep='\t', index=False, header=False)
    print(f"Fragment file saved to {output_dir}")
    return df


In [12]:
scdata = sc.read_h5ad('/home/rsun@ZHANGroup.local/sly_data/grn_momics/data/filter/m_atac.h5ad')
scdata

AnnData object with n_obs × n_vars = 2445 × 142746
    var: 'gene_ids', 'feature_types', 'genome', 'interval', 'm_peaks'
    uns: 'atac'

## cyhm

In [3]:
scdata = sc.read_h5ad('/home/rsun@ZHANGroup.local/sly_data/grn_momics/data/filter/cy_atac.h5ad')
multi_data = sc.read_h5ad('/home/rsun@ZHANGroup.local/sly_data/grn_momics/multi_rna_raw.h5ad')
print(scdata.shape, multi_data.shape)

(5363, 130673) (21723, 9911)


In [6]:
mutual_idx = np.intersect1d(multi_data.obs.index.values, scdata.obs.index.values)
print(len(mutual_idx))

scdata = scdata[mutual_idx,:]
multi_data = multi_data[mutual_idx,:]

scdata.obs = multi_data.obs.copy()

sel_id = [True]*scdata.shape[0]
sel_id = np.array(sel_id)
# filter cluster with less than 20 cells
for key in scdata.obs.leiden.unique():
    if (scdata.obs.leiden == key).sum() < 20:
        sel_id[scdata.obs.leiden == key] = False
        print(f'{key} has cell number {(scdata.obs.leiden == key).sum() }, filtered')
scdata = scdata[sel_id,:]
print(scdata.shape)

5347
3 has cell number 3, filtered
(5344, 130673)


In [14]:
chr = []
start = []
end = []

for ele in scdata.var.loc[:,'m_peaks'].values:
    a,b,c = ele.split('_')
    chr.append(a)
    start.append(int(b))
    end.append(int(c)) 

scdata.var['chr_m'] = chr
scdata.var['start_m'] = start
scdata.var['end_m'] = end


/tmp/ipykernel_2358803/3147861430.py:11: ImplicitModificationWarning: Trying to modify attribute `.var` of view, initializing view as actual.
  scdata.var['chr_m'] = chr


In [ ]:
#scdata.write('/home/rsun@ZHANGroup.local/sly_data/grn_momics/data/filter/cy_atac.h5ad')

In [3]:
scdata = sc.read_h5ad('/home/rsun@ZHANGroup.local/sly_data/grn_momics/data/filter/cy_atac.h5ad')

In [4]:
scdata.X

<5344x130673 sparse matrix of type '<class 'numpy.float32'>'
	with 42212423 stored elements in Compressed Sparse Row format>

In [ ]:
#bgzip -c fragment.tsv > fragment.tsv.gz
#tabix -p bed fragment.tsv.gz

In [6]:
df = adata_to_count_fragments_parallel(scdata, output_dir = 'cyhm_fragments', chr = 'chr_m', start = 'start_m', end = 'end_m', n_cpu=64)

Processing fragments: 100%|██████████| 64/64 [02:12<00:00,  2.07s/it] 


NameError: name 'output_file' is not defined

In [7]:
df

NameError: name 'df' is not defined